In [ ]:
import pandas as pd

# Load Data

In [ ]:
# data = pd.read_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\tonghopdata.csv')

data = pd.read_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\GSE183635_TEP_Count_Matrix.csv')
# name_gene =  pd.read_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\183635\Human_CH38p13annot.csv', low_memory=False    )


In [ ]:
data

In [ ]:
import numpy as np

# Minimal TMM normalization adapted from edgeR logic
# Returns CPM and factors scaled so geometric mean of factors is 1.
def compute_tmm_cpm(counts_df, trim_M=0.3, trim_A=0.05, ref_sample=None):
    counts = counts_df.copy()
    lib_sizes = counts.sum(axis=0)

    # Pick reference sample: closest library size to median if none provided
    if ref_sample is None:
        ref_sample = (lib_sizes - lib_sizes.median()).abs().idxmin()
    ref_counts = counts[ref_sample]
    lib_ref = lib_sizes[ref_sample]

    factors = {}

    def tmm_factor(test_counts, lib_test):
        mask = (test_counts > 0) & (ref_counts > 0)
        if mask.sum() == 0:
            return 1.0

        test = test_counts[mask]
        ref = ref_counts[mask]

        logR = np.log2((test / lib_test) / (ref / lib_ref))
        logA = 0.5 * np.log2((test / lib_test) * (ref / lib_ref))
        weights = 1.0 / ((1.0 / test) + (1.0 / ref))

        # Trim extremes on M and A
        m_low, m_high = np.quantile(logR, [trim_M / 2, 1 - trim_M / 2])
        a_low, a_high = np.quantile(logA, [trim_A / 2, 1 - trim_A / 2])
        keep = (logR >= m_low) & (logR <= m_high) & (logA >= a_low) & (logA <= a_high)

        if keep.sum() == 0:
            return 1.0

        weighted_M = np.average(logR[keep], weights=weights[keep])
        return 2 ** weighted_M

    for col in counts.columns:
        factors[col] = tmm_factor(counts[col], lib_sizes[col])

    factors = pd.Series(factors)
    factors = factors / np.exp(np.log(factors).mean())  # rescale to geometric mean 1
    effective_lib_sizes = lib_sizes * factors

    cpm = counts.divide(effective_lib_sizes, axis=1) * 1_000_000
    return cpm, factors, effective_lib_sizes

In [ ]:
# Apply TMM normalization → CPM → log2(CPM+1)
count_cols = data.columns.drop('Unnamed: 0')
counts_numeric = data[count_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# TMM factors trimmed at 30% for M and 5% for A (edgeR defaults)
tmm_cpm, tmm_factors, tmm_libsizes = compute_tmm_cpm(
    counts_numeric,
    trim_M=0.3,
    trim_A=0.05,
)

tmm_log2 = np.log2(tmm_cpm + 1)

# Reattach gene column and use the normalized values for downstream steps
tmm_log2.insert(0, 'Unnamed: 0', data['Unnamed: 0'])
raw_counts = data.copy()
data = tmm_log2

tmm_log2.head()

In [ ]:
data

In [ ]:
data.drop_duplicates(inplace=True)

In [ ]:
# Transpose so samples become rows
data_transposed = data.set_index('Unnamed: 0').T
data_transposed

In [ ]:
# data_transposed.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\otherdata can be merged\raw.csv', index=True )

In [ ]:
data_transposed['cancer_type'] = (
    data_transposed['cancer_type']
    .replace('Other/Unknown', 'Asymptomatic_Controls')
)


In [ ]:
data_transposed['cancer_type'].value_counts()

In [ ]:
# Extract the names of the remaining 338
remaining_unknowns = [c for c in data.columns if categorize_tumor_type(c) == 'Other/Unknown']

# Look for patterns

print(pd.Series(remaining_unknowns).head(44))

In [ ]:
data_transposed = data_transposed[
    data_transposed['cancer_type'] != 'Symptomatic_Control'
]

In [ ]:
data_transposed.reset_index(drop=True, inplace=True)


In [ ]:
data_transposed

In [ ]:
# 4) Filter low-expression genes (expressed in ≥10% of samples)
min_samples = len(data_transposed) * 0.01
gene_cols = data_transposed.columns.drop('cancer_type')
expressed_genes_mask = (data_transposed[gene_cols] > 0).sum(axis=0) >= min_samples
data_filtered = data_transposed.loc[:, expressed_genes_mask.index[expressed_genes_mask]]
data_filtered['cancer_type'] = data_transposed['cancer_type']


In [ ]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier


In [ ]:
from sklearn.preprocessing import MinMaxScaler

In [ ]:
data_normalized = data_filtered.copy()

In [ ]:
# # # 5) MinMax normalize gene expression values
# # data_normalized = data_filtered.copy()

# # Get gene columns (exclude cancer_type)
# gene_cols = data_normalized.columns.drop('cancer_type')

# # Initialize MinMaxScaler
# scaler = MinMaxScaler()

# # Fit and transform only the gene expression columns
# data_normalized[gene_cols] = scaler.fit_transform(data_filtered[gene_cols])

# # Keep cancer_type column
# data_normalized['cancer_type'] = data_filtered['cancer_type']


In [ ]:
data_normalized

In [ ]:
# 6) Split features/labels
X = data_normalized.drop(columns=['cancer_type'])
y = data_normalized['cancer_type']


In [ ]:
X

In [ ]:
# y.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Multi\data\y_multi_ori.csv', index=False)

In [ ]:
# X.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Multi\data\normalized_X_multi_log.csv', index=False)

In [ ]:
# X.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\X_normalized_binary_log.csv', index=False)

In [ ]:
y = y.apply(lambda x: 1 if x != 'Asymptomatic_Controls' and x != 'Symptomatic_Control' else 0)

In [ ]:
y.value_counts()

In [ ]:
# y.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\y_normalized_binary.csv', index=False)

# embedded RF with stratify k fold

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_validate
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

# Set random state for reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

In [ ]:
# X = pd.read_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\X_normalized_binary_log.csv", low_memory=False)
# y = pd.read_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\y_normalized_binary.csv", low_memory=False).values.ravel()


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)

In [ ]:
# X_train.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_train.csv", index= False)
# X_test.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_test.csv", index= False)
# y_train.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\y_train.csv", index= False)
# y_test.to_csv(r"D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\y_test.csv", index= False)


In [ ]:

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_train)
gene_names = X_train.columns.values

In [ ]:
print("  This may take a few minutes...")

# Parameter grid for RandomForest
param_grid_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [10, 20, 30, None],
    'min_samples_split': [2, 5]
}

# RandomForest classifier
rf = RandomForestClassifier(
    random_state=RANDOM_STATE,
    class_weight='balanced',
    n_jobs=-1
)

# Stratified K-Fold
cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# Grid search
grid_search_rf = GridSearchCV(
    rf,
    param_grid_rf,
    cv=cv_inner,
    scoring='roc_auc',
    n_jobs=-1,
    verbose=0
)

grid_search_rf.fit(X_scaled, y_train)

# Get feature importances
best_rf = grid_search_rf.best_estimator_
feature_importances = best_rf.feature_importances_

### Ranking all genes by RandomForest importance...

In [ ]:
# Create dataframe with ALL genes and their importance scores
rf_df = pd.DataFrame({
    'gene': gene_names,
    'importance': feature_importances
})
rf_df = rf_df.sort_values('importance', ascending=False)

In [ ]:
rf_df

In [ ]:
rf_df.to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\feature_importance.csv', index=False)

In [ ]:
all_features = (
    rf_df.sort_values('importance', ascending=False)
    .head(5440)['gene']
    .tolist()
)

In [ ]:
X_train[all_features]

In [ ]:
X_test[all_features]

In [ ]:
X_train[all_features].to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_train.csv', index=False)
X_test[all_features].to_csv(r'D:\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python\Binary\data\data ranked sk rf\X_test.csv', index=False)

In [ ]:
# import matplotlib.pyplot as plt
# plt.figure(figsize=(6, 6))
# plt.barh(importance_df['Feature'], importance_df['Importance'], color='blue')
# plt.xlabel('Importance Score')
# plt.ylabel('Feature')
# plt.title('Feature Importance Scores from Random Forest')
# plt.gca().invert_yaxis()  # To display the highest importance at the top
# plt.show()

# Start ML


In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

In [30]:
X_train = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\ranked rf\X_train_rF_DEG.csv",low_memory = False)
X_test = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\ranked rf\X_test_rF_DEG.csv",low_memory = False)

y_train = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\ranked rf\y_train_rF_DEG.csv",low_memory = False)
y_test = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\ranked rf\y_test_rF_DEG.csv",low_memory = False)

In [ ]:
# X_train = X_train.iloc[:,:300]
# # 0.00055

In [ ]:
# X_test = X_test.iloc[:,:300]

In [ ]:
from sklearn.model_selection import StratifiedKFold

In [ ]:
from xgboost import XGBClassifier

In [ ]:
from sklearn.preprocessing import StandardScaler
scale = StandardScaler()
x_train_sc = scale.fit_transform(X_train)
x_test_sc = scale.transform(X_test)

In [ ]:
x_train_sc

In [ ]:
list_feature = list(X_train.columns)

In [ ]:
ranks = []
for i in range(len(list_feature)):
    arr = []
    for j in range(i+1):
        arr.append(list_feature[j])
    ranks.append(arr)

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC

pipe = Pipeline([
    ("smote", SMOTE(k_neighbors=3, random_state=42)),
    ("svm", SVC(
        kernel="rbf",
        class_weight="balanced",  
        probability=True,
        random_state = 42
    ))
])

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from sklearn.metrics import confusion_matrix,roc_auc_score
def cfm(y_test,y_pred,y_prob):
    auc = roc_auc_score(y_test,y_prob)
    tn,fp,fn,tp = confusion_matrix(y_test,y_pred).ravel()
    acc = accuracy_score(y_test,y_pred)
    recall = tp/(tp+fn)
    precision = tp/(tp+fp)
    specificity = tn/(tn+fp)
    sensitivity = tp / (tp+fn)
    f1_score = round((2*recall*precision)/(recall+precision),3)
    return np.array([acc,recall,precision,specificity,sensitivity,f1_score,auc])

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.neural_network import MLPClassifier
from skopt import BayesSearchCV
from skopt.space import Real, Categorical, Integer # Skopt uses specialized spaces
from sklearn.metrics import accuracy_score
import numpy as np

# bayesnian optimze

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK
import numpy as np

In [ ]:

# 1. Define the TPE search space
# Converting your parameter lists into hyperopt's hp.choice distributions
space = {
    'C': hp.choice('C', [0.01, 0.1, 1, 10]),
    'kernel': hp.choice('kernel', ['linear', 'poly', 'rbf', 'sigmoid']),
    'gamma': hp.choice('gamma', ['scale', 'auto', 1e-4, 1e-3, 0.01]),
    'degree': hp.choice('degree', [2, 3]),
    'coef0': hp.choice('coef0', [0, 0.1, 0.5, 1])
}

acc_max_rank = None
acc_max = -np.inf
result = []

for i, rank in enumerate(ranks[100:], start=100):
    
    # Isolate the data for the current feature rank iteration
    X_current_train = x_train_sc[:, :i+1]
    y_current_train = y_train['cancer_type']
    X_current_test = x_test_sc[:, :i+1]
    
    # 2. Define the objective function for Hyperopt to minimize
    def objective(params):
        model = SVC(
            C=params['C'],
            kernel=params['kernel'],
            gamma=params['gamma'],
            degree=params['degree'],
            coef0=params['coef0'],
            random_state=122, 
            probability=True
        )
        
        # Calculate 5-fold cross-validation accuracy
        # This replaces the internal cv=5 mechanism of GridSearchCV
        score = cross_val_score(model, X_current_train, y_current_train, cv=5, scoring='accuracy', n_jobs=-1).mean()
        
        # Hyperopt minimizes the objective function, so we must return the negative accuracy
        return {'loss': -score, 'status': STATUS_OK}

    # 3. Run the TPE optimization
    trials = Trials()
    best_hyperparams = fmin(
        fn=objective,
        space=space,
        algo=tpe.suggest,
        max_evals=50, # Set the number of iterations you want TPE to explore
        trials=trials,
        rstate=np.random.default_rng(11)
    )
    
    # 4. Map the output indices back to the actual parameter values
    # Note: When using hp.choice, fmin returns the index of the best option, not the value itself.
    C_choices = [0.01, 0.1, 1, 10]
    kernel_choices = ['linear', 'poly', 'rbf', 'sigmoid']
    gamma_choices = ['scale', 'auto', 1e-4, 1e-3, 0.01]
    degree_choices = [2, 3]
    coef0_choices = [0, 0.1, 0.5, 1]

    best_parameters = {
        'C': C_choices[best_hyperparams['C']],
        'kernel': kernel_choices[best_hyperparams['kernel']],
        'gamma': gamma_choices[best_hyperparams['gamma']],
        'degree': degree_choices[best_hyperparams['degree']],
        'coef0': coef0_choices[best_hyperparams['coef0']]
    }

    # 5. Train the final model with the best parameters
    best_model = SVC(**best_parameters, random_state=122, probability=True)
    best_model.fit(X_current_train, y_current_train)
    
    # 6. Evaluate the model predictions
    y_pred = best_model.predict(X_current_test)
    y_prob = best_model.predict_proba(X_current_test)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    evaluate = cfm(y_test, y_pred, y_prob) # Assuming cfm is your custom evaluation function
    
    result.append([
        rank, 
        '%.2f'%(evaluate[0]*100)+'%',
        '%.2f'%(evaluate[1]*100)+'%',
        '%.2f'%(evaluate[2]*100)+'%',
        '%.2f'%(evaluate[3]*100)+'%',
        '%.2f'%(evaluate[4]*100)+'%',
        '%.2f'%(evaluate[5]*100)+'%',
        '%.2f'%(evaluate[6]*100)+'%',
        best_parameters
    ])

In [ ]:
df_result = pd.DataFrame(result)
df_result.columns = ['Gene','ACC','Recall','Precision','Sp','Sn','f1_Score','AUC','Parameter']
df_result

In [91]:

save_path = r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\sfs\SVM_DEG_on_training_23_4.csv"
df_result = pd.read_csv(save_path, low_memory =False)


df_result2 = pd.read_csv(r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\sfs\SVM_DEG_on_training(test_selected)_23_4.csv", low_memory=False)


# Feature select sfs

In [92]:
df_result = df_result.iloc[:198]
df_result2 = df_result2.iloc[:198]

In [100]:
for col in ["ACC", "Recall", "Precision", "Sp", "Sn", "f1_Score", "AUC"]:
    df_result[col] = df_result[col].str.rstrip("%").astype(float)
for col in ["ACC", "Recall", "Precision", "Sp", "Sn", "f1_Score", "AUC"]:
    df_result2[col] = df_result2[col].str.rstrip("%").astype(float)

In [109]:
df_result

,Gene,ACC,Recall,Precision,Sp,Sn,f1_Score,AUC,Parameter,BA
0,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",78.02,75.96,95.94,86.56,75.96,84.76,87.65,"{'C': 0.1, 'coef0': 0.1, 'degree': 2, 'gamma':...",81.260
1,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",76.10,72.96,96.60,89.11,72.96,83.08,87.75,"{'C': 10, 'coef0': 0.5, 'degree': 3, 'gamma': ...",81.035
2,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",78.14,76.11,95.97,86.55,76.11,84.86,88.18,"{'C': 0.1, 'coef0': 0.5, 'degree': 3, 'gamma':...",81.330
3,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",77.21,74.42,96.56,88.79,74.42,84.00,88.49,"{'C': 0.1, 'coef0': 0.5, 'degree': 2, 'gamma':...",81.605
4,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",79.32,76.88,96.84,89.44,76.88,85.65,89.56,"{'C': 10, 'coef0': 0.1, 'degree': 3, 'gamma': ...",83.160
...,...,...,...,...,...,...,...,...,...,...
193,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",87.74,88.09,96.42,86.24,88.09,92.05,93.75,"{'C': 1, 'coef0': 0.5, 'degree': 2, 'gamma': '...",87.165
194,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",87.86,88.25,96.42,86.24,88.25,92.14,93.76,"{'C': 1, 'coef0': 0.5, 'degree': 2, 'gamma': '...",87.245
195,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",87.86,88.17,96.50,86.56,88.17,92.13,93.75,"{'C': 1, 'coef0': 0.5, 'degree': 2, 'gamma': '...",87.365
196,"['ENSG00000111196', 'ENSG00000227165', 'ENSG00...",87.74,88.02,96.50,86.56,88.02,92.05,93.73,"{'C': 1, 'coef0': 0.5, 'degree': 2, 'gamma': '...",87.290


In [102]:
# 1. Apply the Standard Definition of Balanced Accuracy
df_result['BA'] = (df_result['Sn'] + df_result['Sp']) / 2

In [101]:
# 1. Apply the Standard Definition of Balanced Accuracy
df_result2['BA'] = (df_result2['Sn'] + df_result2['Sp']) / 2

In [ ]:
# Use '&' to ensure BOTH conditions are met
sp_threshold = 0
auc_threshold = 80.0
pre = 0
# This creates a strict filter
mask = (df_result["Sp"] >= sp_threshold) & (df_result["AUC"] >= auc_threshold) & (df_result["Precision"] >= pre)
filtered_df = df_result[mask]

if not filtered_df.empty:
    best_row = filtered_df.loc[filtered_df["BA"].idxmax()]
    print("Best Model Meeting Strict Criteria:")
    print(best_row)
else:
    print(f"No models found with Sp > {sp_threshold} and AUC > {auc_threshold}.")
    # Optional: see what the highest Sp available is
    print(f"Max Sp in data: {df_result['Sp'].max()}")

In [116]:
df_result.loc[df_result["AUC"].idxmax()]
# balanced accuracy 195-87.365
# AUC  318-94.2
# 
#  

Gene         ['ENSG00000111196', 'ENSG00000227165', 'ENSG00...
ACC                                                       87.8
Recall                                                    88.4
Precision                                                96.18
Sp                                                       85.29
Sn                                                        88.4
f1_Score                                                 92.11
AUC                                                      93.92
Parameter    {'C': 1, 'coef0': 0.5, 'degree': 2, 'gamma': '...
BA                                                      86.845
Name: 197, dtype: object

In [55]:
df_result.loc[182]

Gene         ['ENSG00000111196', 'ENSG00000227165', 'ENSG00...
ACC                                                      87.31
Recall                                                   87.71
Precision                                                96.23
Sp                                                       85.61
Sn                                                       87.71
f1_Score                                                 91.76
AUC                                                      93.53
Parameter    {'C': 1, 'coef0': 0.1, 'degree': 2, 'gamma': '...
BA                                                       86.66
Name: 182, dtype: object

In [115]:
df_result2.loc[194]

Gene         ['ENSG00000111196', 'ENSG00000227165', 'ENSG00...
ACC                                                      86.63
Recall                                                   87.42
Precision                                                95.64
Sp                                                       83.33
Sn                                                       87.42
f1_Score                                                  91.3
AUC                                                      94.17
Parameter    {'C': 1, 'coef0': 0.5, 'degree': 2, 'gamma': '...
BA                                                      85.375
Name: 194, dtype: object

In [113]:
df_result2.loc[df_result2["ACC"].idxmax()]

Gene         ['ENSG00000111196', 'ENSG00000227165', 'ENSG00...
ACC                                                      87.62
Recall                                                   88.96
Precision                                                95.39
Sp                                                       82.05
Sn                                                       88.96
f1_Score                                                  92.1
AUC                                                      94.46
Parameter    {'C': 1, 'coef0': 0.5, 'degree': 2, 'gamma': '...
BA                                                      85.505
Name: 180, dtype: object

In [24]:
best_genes = df_result.loc[df_result["AUC"].idxmax(), "Gene"]

In [27]:
import ast

# Convert string → list
best_genes = ast.literal_eval(best_genes)

ValueError: malformed node or string: ['ENSG00000111196', 'ENSG00000185825', 'ENSG00000130948', 'ENSG00000227165', 'ENSG00000240225', 'ENSG00000142089', 'ENSG00000249307', 'ENSG00000230773', 'ENSG00000172757', 'ENSG00000163359', 'ENSG00000142856', 'ENSG00000133742', 'ENSG00000069535', 'ENSG00000173083', 'ENSG00000130429', 'ENSG00000103512', 'ENSG00000153046', 'ENSG00000005961', 'ENSG00000213465', 'ENSG00000103316', 'ENSG00000137076', 'ENSG00000105701', 'ENSG00000097021', 'ENSG00000105220', 'ENSG00000197459', 'ENSG00000120159', 'ENSG00000163950', 'ENSG00000196924', 'ENSG00000143226', 'ENSG00000267316', 'ENSG00000134317', 'ENSG00000250334', 'ENSG00000167468', 'ENSG00000179526', 'ENSG00000158578', 'ENSG00000151023', 'ENSG00000165949', 'ENSG00000167645', 'ENSG00000101335', 'ENSG00000112290', 'ENSG00000091317', 'ENSG00000100243', 'ENSG00000104341', 'ENSG00000153064', 'ENSG00000149243', 'ENSG00000102760', 'ENSG00000023697', 'ENSG00000251399', 'ENSG00000105402', 'ENSG00000123505', 'ENSG000001

In [28]:
X_train

,ENSG00000227165,ENSG00000130948,ENSG00000111196,ENSG00000240225,ENSG00000005961,ENSG00000103512,ENSG00000142089,ENSG00000103316,ENSG00000069535,ENSG00000142856,...,ENSG00000100385,ENSG00000189067,ENSG00000110719,ENSG00000007392,ENSG00000033050,ENSG00000170876,ENSG00000104518,ENSG00000158435,ENSG00000115421,ENSG00000125844
0,6.830194,3.421733,5.313933,5.029751,12.739048,5.847213,5.163708,6.060632,5.903704,6.396570,...,2.977562,2.977562,3.421733,2.977562,2.977562,2.977562,4.957158,2.977562,3.740992,2.977562
1,5.657219,2.977562,4.567253,4.937442,12.818652,5.554333,6.938392,5.273281,5.619536,7.267184,...,2.977562,2.977562,4.012291,3.716893,3.552716,2.977562,3.961204,3.311087,2.977562,2.977562
2,7.332512,4.741644,6.504281,6.350295,12.668418,7.662607,5.189230,5.001946,5.770137,8.019397,...,4.229133,4.655881,4.409050,4.161411,4.292595,3.922951,3.826072,3.826072,3.922951,3.581808
3,7.701091,5.640744,7.099868,7.587544,12.174109,7.014400,6.064303,5.880896,3.690989,6.875797,...,2.977562,4.976052,6.147726,2.977562,4.189897,2.977562,3.484574,3.690989,2.977562,3.484574
4,6.625362,4.777392,2.977562,2.977562,12.355583,5.397404,6.279014,5.989692,2.977562,6.404094,...,2.977562,4.288787,8.166124,4.777392,2.977562,5.624376,5.397404,2.977562,2.977562,6.404094
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1610,7.278084,4.727932,6.566049,6.827255,13.250418,6.910702,4.750060,3.952941,5.109498,8.081787,...,5.307998,4.452481,5.720256,4.161529,4.233613,4.083676,4.083676,3.952941,3.798148,4.198224
1611,5.789434,4.181500,5.438894,5.055621,11.495049,7.121808,6.730757,2.977562,5.141714,7.998309,...,2.977562,4.755728,9.148967,5.736876,5.625344,5.141714,5.370572,5.298584,3.685866,5.503925
1612,7.212175,5.211901,5.934437,5.383968,12.435184,7.225306,4.671672,4.044359,6.181557,7.638879,...,5.076540,5.248143,6.539943,4.878589,3.875994,2.977562,4.310049,4.521444,3.571079,4.229148
1613,6.291087,2.977562,2.977562,2.977562,12.095441,5.526078,6.916693,5.526078,2.977562,5.830206,...,2.977562,4.566351,7.253598,5.526078,2.977562,2.977562,4.566351,5.133454,2.977562,2.977562


In [26]:
# Select features
X_train = X_train[best_genes]
X_test  = X_test[best_genes]

KeyError: "['ENSG00000185825', 'ENSG00000172757', 'ENSG00000130429', 'ENSG00000120159', 'ENSG00000163950', 'ENSG00000167468', 'ENSG00000151023', 'ENSG00000167645', 'ENSG00000153064', 'ENSG00000023697', 'ENSG00000105402', 'ENSG00000139970', 'ENSG00000184178', 'ENSG00000067167', 'ENSG00000198836', 'ENSG00000102974', 'ENSG00000143162', 'ENSG00000002586', 'ENSG00000077549', 'ENSG00000130066', 'ENSG00000143353', 'ENSG00000108010', 'ENSG00000113282', 'ENSG00000082146', 'ENSG00000126458', 'ENSG00000155158', 'ENSG00000227507', 'ENSG00000175470', 'ENSG00000143742', 'ENSG00000100060', 'ENSG00000060138', 'ENSG00000162704', 'ENSG00000023191', 'ENSG00000108405', 'ENSG00000100711', 'ENSG00000187699', 'ENSG00000068308', 'ENSG00000123106', 'ENSG00000128708', 'ENSG00000151247', 'ENSG00000123124', 'ENSG00000125952', 'ENSG00000138449', 'ENSG00000136404', 'ENSG00000136717', 'ENSG00000137492', 'ENSG00000028839', 'ENSG00000176171', 'ENSG00000147099', 'ENSG00000153113', 'ENSG00000158869', 'ENSG00000227355', 

In [ ]:
X_train

# Test set

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from xgboost import XGBClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import StratifiedKFold
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

In [ ]:
# from sklearn.preprocessing import StandardScaler
# scale = StandardScaler()
# X_train = scale.fit_transform(X_train)
# X_test = scale.transform(X_test)

In [ ]:
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE
from sklearn.svm import SVC

pipe = Pipeline([
    ("smote", SMOTE(k_neighbors=3, random_state=42)),
    ("svm", SVC(
        kernel="rbf",
        class_weight="balanced",  
        probability=True,
        random_state = 42
    ))
])
svm_model = SVC(
    class_weight='balanced',
    random_state= 42,
    probability=True,
    verbose=0
)

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

In [ ]:
from sklearn.model_selection import GridSearchCV


param_grid = {
    # "svm__C": [0.1, 1, 10, 100, 1000],
    # "svm__kernel": ["linear", "rbf"],
    # "svm__gamma": ["scale", "auto", 0.001, 0.01, 0.1, 1],
    
    "svm__C": [10],
    "svm__kernel": ["rbf"],
    "svm__gamma": ["auto"],
    
    # "C": [10],
    # "kernel": ["rbf"],
    # "gamma": ["scale", "auto", 0.01],
}

grid_search = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    cv=skf,
    scoring="accuracy",
    n_jobs=-1,
    refit=True,
    verbose=1,
 )


In [ ]:
grid_search

In [ ]:
X_train

In [ ]:
import time
start_time = time.time()
grid_search.fit(X_train, y_train['cancer_type'])
end_time = time.time()

In [ ]:
import pandas as pd
import numpy as np

# Lấy kết quả từ GridSearch
results = grid_search.cv_results_

# Tạo DataFrame
df_results = pd.DataFrame(results)

# Tính phương sai từ std (vì sklearn cung cấp std)
df_results['var_test_score'] = df_results['std_test_score'] ** 2

# Chọn các cột quan trọng
summary_table = df_results[[
    'params',
    'mean_test_score',
    'std_test_score',
    'var_test_score',
    'rank_test_score'
]]

# Sắp xếp theo mean_test_score giảm dần
summary_table = summary_table.sort_values(by='mean_test_score', ascending=False)


In [ ]:
summary_table.head(10)

In [ ]:
best_params = grid_search.best_params_

In [ ]:
best_accuracy = grid_search.best_score_

In [ ]:
best_params

In [ ]:
best_accuracy

In [ ]:
# Calculate training time
training_time = end_time - start_time

In [ ]:
y_pred = grid_search.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

### Reslut for testing set

In [ ]:
from sklearn.metrics import roc_curve, auc

In [ ]:
print("Training time:", training_time, "seconds")
print(classification_report( y_test['cancer_type'],y_pred,digits=6))


### Plot Figure

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sn

In [ ]:
mat_imp = confusion_matrix(y_test['cancer_type'],y_pred)
sn.heatmap(mat_imp,square=True, annot=True, fmt='d', cbar=False)
plt.xlabel("Actual Values")
plt.ylabel("Predicted Values")

In [ ]:
from sklearn.metrics import roc_curve, auc
y_scores = grid_search.predict_proba(X_test)
fpr, tpr, thresholds = roc_curve(y_test['cancer_type'], y_scores[:, 1])
roc_auc = auc(fpr, tpr)
roc_auc

In [ ]:
plt.title('ROC for RF Binary Classification')
plt.figure(figsize=(10,8))
plt.plot(fpr,tpr, marker = '*',color='darkorange',label = 'AUC = %0.6f' % roc_auc)
plt.legend(loc = 'lower right')
plt.plot([0, 1], [0, 1],'--')
plt.xlim([0, 1])
plt.ylim([0, 1])
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.title('ROC Curve of RandomForest')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6,6))
plt.title('ROC of svm')
plt.plot(fpr, tpr, color='darkblue', label='AUC = %0.6f' % roc_auc)
plt.legend(loc='lower right', fontsize=20)
plt.plot([0, 1], [0, 1], linestyle='--')
plt.axis('scaled')
plt.ylabel('True Positive Rate', fontsize=14)
plt.xlabel('False Positive Rate', fontsize=14)
plt.show()
